### V1 — Video Inspector
Write a script that loads a video file and prints a clean summary:
```
FPS           : 30.0
Resolution    : 1280 x 720
Total Frames  : 900
Duration      : 30.00 seconds
Codec FourCC  : (print the raw int from CAP_PROP_FOURCC)
```
> **Constraint**: Calculate duration yourself from total frames and FPS — don't look for a built-in duration property.

In [1]:
import cv2 as cv

# Open your video file or stream
cap = cv.VideoCapture("outpy.avi")

if cap.isOpened():
    # Fetch values (cast to int where appropriate)
    fps = cap.get(cv.CAP_PROP_FPS)
    frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
    codecFourCC = int(cap.get(cv.CAP_PROP_FOURCC))

    # FIX: Divide total frames by FPS to get duration in seconds
    duration = total_frames / fps if fps > 0 else 0

    # Print matching your requested format
    print(f"# FPS           : {fps:.1f}")
    print(f"# Resolution    : {frame_width} x {frame_height}")
    print(f"# Total Frames  : {total_frames}")
    print(f"# Duration      : {duration:.2f} seconds")
    print(f"# Codec FourCC  : {codecFourCC}")

    cap.release()

# FPS           : 10.0
# Resolution    : 640 x 480
# Total Frames  : 125
# Duration      : 12.50 seconds
# Codec FourCC  : 1196444237


### V2 — Frame Dumper
Open a video. Save **only** these specific frames as JPEGs into an `output/` folder:
- Frame 0 (first frame)
- Frame at the midpoint
- Last frame

Filename format: `frame_000000.jpg`, `frame_000450.jpg` etc.

> **Think about**: How do you jump to a specific frame without reading every frame before it? Look at `CAP_PROP_POS_FRAMES`.

In [ ]:
import cv2 as cv
import numpy as np

# Open your video file or stream
cap = cv.VideoCapture("outpy.avi")

if not cap.isOpened():
    print("Error: Opening Video")
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = int(cap.get(cv.CAP_PROP_FPS))
duration = total_frames / fps if fps > 0 else 0

images_obj = {}
while (True):
    ret,frame = cap.read()

    if not ret:
        break
    
    curr_pos = cap.get(cv.CAP_PROP_POS_FRAMES)/10
    if curr_pos == 0.1 or curr_pos == duration // 2 or curr_pos == duration:
        images_obj[f'frame_{curr_pos}.jpg'] = frame
    

cap.release()
cv.destroyAllWindows()

for file_name,frame in images_obj.items():
    success = cv.imwrite(file_name,frame)
    if success:
        print(f"Image saved successfully {file_name}!")
    else:
        print(f"Failed to save the image")
    print()

### V3 — The Frame Counter Overlay
Play a video in a window. On every frame, draw the current frame number and timestamp (in seconds, 2 decimal places) in the top-left corner.

Example overlay: `Frame: 042 | Time: 1.40s`

> **Constraint**: Don't use a separate counter variable — derive the frame number from a `cap.get()` call on each iteration.

In [ ]:
import cv2 as cv
import numpy as np

# Open your video file or stream
cap = cv.VideoCapture('outpy.avi')

if not cap.isOpened():
    print("Error: Could not open the video source.")
    exit()

while cap.isOpened():
    ret,frame = cap.read()
    current_frame = int(cap.get(cv.CAP_PROP_POS_FRAMES))

    # Get current time position in milliseconds
    msec_time = cap.get(cv.CAP_PROP_POS_MSEC)
    
    # Convert milliseconds to seconds
    seconds_time = round(msec_time / 1000.0,2)
    
    print(f"Current Frame: {current_frame}")
    print(f"Current Time: {seconds_time}")

    if not ret:
        print("Video playback finished.")
        break
    cv.putText(frame, f"Frame: {current_frame} | Time: {seconds_time}s", (10,30), cv.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 1, cv.LINE_AA)

    cv.imshow('Frame',frame)

    if cv.waitKey(25) & 0xFF == ord('q'):
        break



cap.release()
cv.destroyAllWindows()

### V4 — Sampler
Write a script that extracts every **Nth frame** from a video and saves them as images. `N` should be a variable at the top of your script you can easily change.

At the end, print:
```
Total frames in video : 900
Frames sampled        : 90
Saved to              : output/
```
> **Real-world use**: This is exactly how you'll feed frames into an ML model without processing every single frame at full FPS.

In [ ]:
import cv2 as cv
import os 

N = int(input("Enter the number of frames you want to sample: "))
# Open your video file or stream
cap = cv.VideoCapture('outpy.avi')

if not cap.isOpened():
    print("Error: Could not open the video source.")
    exit()
framesObj = {}
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))

i = 1
while (True):
    ret,frame = cap.read()


    if i <= N:
        framesObj[f"frame_{i}.jpg"] = frame 
        i += 1
    else:
        break
    if not ret:
        print("Video playback finished.")
        break




cap.release()
cv.destroyAllWindows()
folder = "output"
for file_name,frame in framesObj.items():
    success = cv.imwrite(os.path.join(folder,file_name),frame)
    if success:
        print(f"Image saved successfully {file_name}!")
    else:
        print(f"Failed to save the image")
    print()

print(f"Total frames in video  : {total_frames}")
print(f"Frames sampled         : {i}")
print("Saved to                : output/")